In [23]:
# This document contains useful functions for experimentation with caterpillars, as well as computations
# that we are currently exploring

In [30]:
%run DNCfundamentals.ipynb
%run helperFunctions.ipynb

In [31]:
# Ouptut: list containing all caterpillars on n vertices in the form of compositions
def generate_caterpillars(n):
    seen = set()
    compositions = list(Compositions(n))
    caterpillar_list = []
    
    for comp in compositions:
        if comp[0] != 1 and comp[-1] != 1:
            reversal = comp[::-1]
            if tuple(reversal) not in seen:
                seen.add(tuple(comp))
                caterpillar_list.append(comp)
    
    return caterpillar_list

In [32]:
# Output: list of all proper caterpillars on n vertices in the form of compositions
def generate_proper_caterpillars(n):
    seen = set()
    compos = list(Compositions(n))
    caterpillar_list = []
    proper_compos = []
    
    for comp in compos:
        if count_non_ones(comp) == len(comp):
            proper_compos.append(comp)
    
    for comp in proper_compos:
        if comp[0] != 1 and comp[-1] != 1:
            reversal = comp[::-1]
            if tuple(reversal) not in seen:
                seen.add(tuple(comp))
                caterpillar_list.append(comp)
    
    return caterpillar_list

In [33]:
# Function that, given a CSF vector, returns an array containing the terms in the CSF 
#     corresponding to partitions of length 1 and 2 less than the leading in the form (mu, c_mu)

def mu_nu_partitions(CSFvector,n):
    partitions = Partitions(n).list()
    leading = get_leading_partition(CSFvector, n)
    mu_nu_partitions = []
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-1 and count_ones(lbda)==0 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-2 and count_ones(lbda)==0 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    return mu_nu_partitions

In [34]:
# Output: a map that sends a caterpillar --> a tuple containing its leading partition, and its mu and nu partitions
#         as defined in the function above
def caterpillars_leading_nu_map(n):
    map = {}
    for comp in generate_caterpillars(n):
        if count_ones(comp)==1:
            C=create_caterpillar(comp)
            C=C.copy()
            lst = list(mu_nu_partitions(CSF_tree(C), n))
            lst.insert(0, tuple(sorted(comp, reverse=True)))
            tpl = tuple(lst)
            map[tuple(comp)]=tpl
    return map

# Input: a composition (comp) representing a caterpillars
# Outputs a dictionary containing edges with multiplicity appearing in a caterpillar with composition comp
def caterpillar_edges(comp):
    edge_map = {}
    for i in range(len(comp)-1):
        lst = sorted([comp[i], comp[i+1]], reverse=True)
        edge = tuple(lst)
        if edge in edge_map.keys():
            curr = edge_map[edge]
            edge_map[edge] = curr+1
        else:
            edge_map[edge]=1
    return edge_map

In [35]:
n=15

mp = caterpillars_leading_nu_map(n)

flipped = {}

for key, value in mp.items():
    tplkey = tuple(key)
    tplvalue=tuple(value)
    
    if tplvalue not in flipped.keys():
        flipped[tplvalue] = tuple([tplkey])

    else:
        lst = list(flipped[tplvalue])
#         print(lst)
        lst.append(tplkey)
        tpl = tuple(lst)
        flipped[tplvalue]=tpl

for key in flipped.keys():
    if len(flipped[key])>1:
        for i in range(len(flipped[key])-1):
            if not caterpillar_edges(flipped[key][i]) == caterpillar_edges(flipped[key][i+1]):
                print(key, '-->', flipped[key])
                break

((3, 3, 2, 2, 2, 2, 1), ([3, 3, 3, 2, 2, 2], 2), ([6, 3, 2, 2, 2], 2), ([5, 3, 3, 2, 2], 5), ([4, 3, 3, 3, 2], 2)) --> ((2, 2, 1, 2, 2, 3, 3), (2, 2, 3, 2, 1, 2, 3))
((4, 3, 3, 2, 2, 1), ([4, 4, 3, 2, 2], 1), ([4, 3, 3, 3, 2], 1), ([7, 4, 2, 2], 1), ([7, 3, 3, 2], 1), ([6, 4, 3, 2], 3), ([5, 4, 4, 2], 1), ([5, 4, 3, 3], 1)) --> ((2, 3, 1, 2, 3, 4), (2, 3, 3, 1, 2, 4))
